# M5-T1: Evaluation Protocol & Experiment Harness

**Owner:** Sanjeewa Narayana  
Shared scaffold so every M5 experiment (T2 email, T3 URL, T7/T8 deep) is **directly comparable**: same metrics, same 5-fold stratified CV, same fixed splits, same results log.

**Run this notebook once to validate, then T2/T3/T7/T8 reuse these helpers.**

## Step 0 — Install + download splits (run once in terminal)
```bash
pip install pandas scikit-learn
mkdir -p data/processed
for f in email_train email_test url_train url_test; do
  aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/$f.csv data/processed/$f.csv --profile lab-user
done
```

In [ ]:
import os
os.environ.setdefault('PYTHONWARNINGS', 'ignore')  # propagates to joblib workers
import warnings
warnings.filterwarnings('ignore')
import time, csv
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

## Step 1 — Configuration (the locked protocol)
Primary metric: **F1** (+ ROC-AUC), because both datasets are imbalanced. CV: 5-fold stratified on TRAIN; final numbers on the held-out TEST split.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
RESULTS = ROOT / 'results' / 'm5_results.csv'
RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

EMAIL_TEXT = 'text_clean'
# link_count & html_ratio are constant 0 on this corpus (URLs/HTML pre-stripped) -> dropped to
# avoid zero-variance scaling noise. They stay in the CSV; just not fed to the model here.
EMAIL_NUM  = ['urgency_score','word_count','avg_word_length']
URL_FEATS  = ['url_length','hostname_length','num_dots','num_hyphens','num_at',
              'num_digits','num_special_chars','has_ip','has_https',
              'num_subdomains','is_shortened']
print('Results log ->', RESULTS)

## Step 2 — Split loaders
`text_clean` NaN is repaired to '' (M4-T7 already did this, but be safe).

In [ ]:
def load_email():
    tr = pd.read_csv(PROC / 'email_train.csv'); te = pd.read_csv(PROC / 'email_test.csv')
    for d in (tr, te):
        d[EMAIL_TEXT] = d[EMAIL_TEXT].fillna('')
    return tr, te

def load_url():
    tr = pd.read_csv(PROC / 'url_train.csv'); te = pd.read_csv(PROC / 'url_test.csv')
    return tr, te

## Step 3 — Metric helper
Handles models with `predict_proba` (LogReg, RF, NB) and `decision_function` (LinearSVC) for ROC-AUC.

In [ ]:
def _proba(pipe, X):
    if hasattr(pipe, 'predict_proba'):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, 'decision_function'):
        return pipe.decision_function(X)
    return None

def scores(y_true, y_pred, y_score=None):
    out = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_true, y_score) if y_score is not None else float('nan'),
    }
    return out

## Step 4 — Pipeline factories
Email = TF-IDF(text) + scaled numerics. URL = scaled numerics (trees pass `scale=False`).
For MultinomialNB use `email_pipeline(clf, tfidf_only=True)` (NB needs non-negative input).

In [ ]:
def email_pipeline(clf, tfidf_only=False):
    parts = [('text', TfidfVectorizer(max_features=20000, ngram_range=(1, 2)), EMAIL_TEXT)]
    if not tfidf_only:
        parts.append(('num', StandardScaler(with_mean=False), EMAIL_NUM))
    return Pipeline([('pre', ColumnTransformer(parts)), ('clf', clf)])

def url_pipeline(clf, scale=True):
    steps = [('scale', StandardScaler())] if scale else []
    steps.append(('clf', clf))
    return Pipeline(steps)

## Step 5 — `run_experiment` (CV + test + logging)
Every model goes through this so results are comparable and auto-logged.

In [ ]:
def _log(row):
    RESULTS.parent.mkdir(parents=True, exist_ok=True)
    cols = ['track','model','accuracy','precision','recall','f1','roc_auc','cv_f1','train_time']
    exists = RESULTS.exists()
    with open(RESULTS, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols)
        if not exists: w.writeheader()
        w.writerow({k: row.get(k, '') for k in cols})

def run_experiment(name, track, pipe, Xtr, ytr, Xte, yte, log=True):
    t0 = time.time()
    cv_f1 = cross_val_score(pipe, Xtr, ytr, cv=CV, scoring='f1', n_jobs=-1).mean()
    pipe.fit(Xtr, ytr)
    train_time = time.time() - t0
    y_pred = pipe.predict(Xte)
    m = scores(yte, y_pred, _proba(pipe, Xte))
    row = {'track': track, 'model': name,
           **{k: round(v, 4) for k, v in m.items()},
           'cv_f1': round(cv_f1, 4), 'train_time': round(train_time, 2)}
    print(f"[{track}] {name}: F1={m['f1']:.4f}  ROC-AUC={m['roc_auc']:.4f}  "
          f"(cv_f1={cv_f1:.4f}, {train_time:.1f}s)")
    if log: _log(row)
    return row

## Step 6 — Validation demo (one model per track)
Subsampled for speed — just to confirm the harness works end-to-end.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# EMAIL demo
etr, ete = load_email()
etr_s = etr.sample(8000, random_state=RANDOM_STATE)
run_experiment('LogReg (demo)', 'email',
               email_pipeline(LogisticRegression(max_iter=1000)),
               etr_s, etr_s['label'], ete, ete['label'])

# URL demo
utr, ute = load_url()
utr_s = utr.sample(50000, random_state=RANDOM_STATE)
run_experiment('RandomForest (demo)', 'url',
               url_pipeline(RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                                   class_weight='balanced'), scale=False),
               utr_s[URL_FEATS], utr_s['label'], ute[URL_FEATS], ute['label'])

print('\nResults logged to', RESULTS)

## How M5-T2 / T3 / T7 / T8 use this harness
```python
# import the helpers (or %run this notebook), then for each candidate:
from sklearn.svm import LinearSVC
etr, ete = load_email()
run_experiment('LinearSVC', 'email', email_pipeline(LinearSVC()), etr, etr['label'], ete, ete['label'])

import xgboost as xgb  # URL track
utr, ute = load_url()
run_experiment('XGBoost', 'url', url_pipeline(xgb.XGBClassifier(n_estimators=300), scale=False),
               utr[URL_FEATS], utr['label'], ute[URL_FEATS], ute['label'])
```
All rows accumulate in `results/m5_results.csv` → feeds M5-T4 (consolidate) and M5-T5 (comparison table).